# 06 · Evidence-first LLM structured extraction

**Spatial Humanities 2026 workshop**

Large language models can represent richer spatial relations than conventional NER, but richer outputs create a larger validation burden. This notebook therefore treats the model as a **proposal generator**, not an authority.

## Learning goals
- inspect a schema-constrained journey prompt;
- distinguish `explicit`, `contextual_inference`, and `missing` fields;
- require a verbatim evidence quotation;
- compute evidence offsets locally rather than trusting model-generated offsets;
- route unsupported, ambiguous, or inferred outputs to review;
- understand why `null` can be a valid scholarly result;
- optionally run the same workflow with a live LLM provider.

> **Key message:** The useful LLM pattern is constrained extraction + evidence + validation, not fluent generation.

In [ ]:
# Independent Colab setup.
import os, sys, json, subprocess, pathlib

REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"
BRANCH = "spatial-humanities-2026"

if not pathlib.Path("spatio-textual").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "spatio-textual"], check=True)
os.chdir("spatio-textual")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print("Ready:", pathlib.Path.cwd())

## 1. Start with a public-safe synthetic narrative

The Q/A-like context matters because an origin can be established in one sentence and a movement described in the next.

In [ ]:
source = (
    "Interviewer: Where were you living? "
    "Narrator: I was in Amsterdam. "
    "The next morning I travelled by train to Brussels to join my sister."
)
print(source)

## 2. Inspect the extraction contract

The prompt deliberately asks the model for a **verbatim evidence quote**, but it does **not** ask the model for character offsets. Offsets are computed by Python after the response returns.

In [ ]:
from spatio_textual.journeys import build_journey_prompt

prompt = build_journey_prompt(source)
print(prompt)

Notice several restrictions:

- a place mention alone is not a journey;
- missing fields must remain `null`;
- contextual inference must be labelled;
- historical names must not be silently modernised;
- evidence must be copied from the source;
- confidence refers to the extraction, not historical truth.

## 3. Deterministic teaching client

To make the workshop reliable without API keys, we first simulate a model response. This is not presented as an empirical LLM result. It is a controlled example for learning the validation machinery.

In [ ]:
from spatio_textual.journeys import JourneyExtractor, validate_runtime_journey

class TeachingClient:
    provider = "teaching"
    model = "deterministic-example"

    def complete_json(self, task, prompt, *, input_text=None):
        return {
            "journeys": [{
                "start_location": "Amsterdam",
                "end_location": "Brussels",
                "transport_mode": "train",
                "date": "The next morning",
                "journey_reason": "to join my sister",
                "evidence_quote": "The next morning I travelled by train to Brussels to join my sister.",
                "explicit_or_inferred": {
                    "start_location": "contextual_inference",
                    "end_location": "explicit",
                    "transport_mode": "explicit",
                    "date": "explicit",
                    "journey_reason": "explicit",
                },
                "confidence": 0.92,
                "notes": ["Origin comes from the immediately preceding narrator statement."],
            }],
            "telemetry": {
                "task": task,
                "backend": "teaching",
                "provider": self.provider,
                "model": self.model,
                "success": True,
            }
        }

extractor = JourneyExtractor(client=TeachingClient())
result = extractor.extract(source, file_id="synthetic-qa-01", seg_id=0)
print(json.dumps(result, indent=2, ensure_ascii=False))

The origin is intentionally marked `contextual_inference`: **Amsterdam** is supported by the previous sentence, not by the movement clause itself. That distinction automatically routes the record to review.

## 4. Verify evidence grounding

The model supplied a quotation. Python found that quotation in the original source and calculated the offsets locally.

In [ ]:
journey = result["journeys"][0]
start = journey["evidence_start_char"]
end = journey["evidence_end_char"]

print("Grounded:", journey["evidence_grounded"])
print("Offsets:", start, end)
print("Source slice:", source[start:end])
print("Quote:", journey["evidence_quote"])
assert source[start:end] == journey["evidence_quote"]
assert validate_runtime_journey(journey, source) == []

This avoids a common provenance failure: a generative model can produce plausible-looking numeric offsets even when they do not correspond to the source. Here, the model never gets that authority.

## 5. What happens when the model invents evidence?

The next client returns a quotation that does not occur in the source. The extraction is retained for audit, but it is not treated as grounded evidence.

In [ ]:
class BadEvidenceClient:
    provider = "teaching"
    model = "bad-evidence-example"

    def complete_json(self, task, prompt, *, input_text=None):
        return {
            "journeys": [{
                "start_location": "Amsterdam",
                "end_location": "Brussels",
                "transport_mode": "train",
                "date": None,
                "journey_reason": None,
                "evidence_quote": "I boarded the train in Amsterdam and arrived in Brussels.",
                "explicit_or_inferred": {
                    "start_location": "explicit",
                    "end_location": "explicit",
                    "transport_mode": "explicit",
                    "date": "missing",
                    "journey_reason": "missing",
                },
                "confidence": 0.99,
            }]
        }

bad = JourneyExtractor(client=BadEvidenceClient()).extract(source, file_id="synthetic-qa-01", seg_id=0)
print(json.dumps(bad["journeys"][0], indent=2, ensure_ascii=False))

Expected audit behaviour:

- `evidence_grounded = false`
- evidence offsets are `null`
- `requires_review = true`
- the review notes explicitly state that the quote is not an exact substring of the source

A high model confidence cannot override failed provenance.

## 6. Schema violations are visible, not silently repaired

Fixed vocabularies such as `explicit | contextual_inference | missing` make unexpected model behaviour testable. Invalid states are converted conservatively and recorded in review notes.

In [ ]:
class InvalidStatusClient:
    provider = "teaching"
    model = "invalid-status-example"

    def complete_json(self, task, prompt, *, input_text=None):
        return {
            "journeys": [{
                "start_location": "Amsterdam",
                "end_location": "Brussels",
                "transport_mode": None,
                "date": "tomorrow",
                "journey_reason": None,
                "evidence_quote": "The next morning I travelled by train to Brussels to join my sister.",
                "explicit_or_inferred": {
                    "start_location": "probably",
                    "end_location": "explicit",
                    "transport_mode": "explicit",
                    "date": "missing",
                    "journey_reason": "missing",
                },
                "confidence": 1.4,
            }]
        }

invalid = JourneyExtractor(client=InvalidStatusClient()).extract(source, file_id="synthetic-qa-01", seg_id=0)
print(json.dumps(invalid["journeys"][0], indent=2, ensure_ascii=False))

The validator should expose several problems rather than hiding them: an unsupported status, a status/value contradiction, and confidence outside the permitted range.

## 7. `null` is a scholarly result

LLMs often create pressure to fill every field in a schema. In historical and humanities research, absence of evidence should remain absence of evidence.

In [ ]:
minimal_source = "I left Cambridge and travelled to London."

class NullFriendlyClient:
    provider = "teaching"
    model = "null-friendly-example"

    def complete_json(self, task, prompt, *, input_text=None):
        return {
            "journeys": [{
                "start_location": "Cambridge",
                "end_location": "London",
                "transport_mode": None,
                "date": None,
                "journey_reason": None,
                "evidence_quote": minimal_source,
                "explicit_or_inferred": {
                    "start_location": "explicit",
                    "end_location": "explicit",
                    "transport_mode": "missing",
                    "date": "missing",
                    "journey_reason": "missing",
                },
                "confidence": 0.95,
            }]
        }

null_result = JourneyExtractor(client=NullFriendlyClient()).extract(minimal_source, file_id="synthetic-null-01", seg_id=0)
print(json.dumps(null_result["journeys"][0], indent=2, ensure_ascii=False))

Nothing is "wrong" with the missing transport, date or reason. The source does not provide them.

> **Null is not an incomplete answer when the source is silent.**

## 8. Summarise representational status

The status distribution itself is analytically useful. It tells us how much of a structured representation was explicit, inferred, absent, or later supplied by a human.

In [ ]:
from spatio_textual.journeys import journey_field_status_counts

counts = journey_field_status_counts(result["journeys"] + null_result["journeys"])
print(json.dumps(counts, indent=2))

## 9. Optional live LLM call

This cell runs only if you deliberately configure a provider. Do **not** paste secrets into notebook cells or print them.

Typical environment variables are provider-specific. The workshop release will document safe Colab secret handling and will include precomputed outputs so no participant is required to make a paid API call.

In [ ]:
RUN_LIVE_LLM = False  # change deliberately

if RUN_LIVE_LLM:
    from spatio_textual.llm import LLMClient

    provider = os.getenv("LLM_PROVIDER", "openai")
    model = os.getenv("LLM_MODEL") or None
    client = LLMClient(provider=provider, model=model)
    live = JourneyExtractor(client=client).extract(source, file_id="synthetic-live-01", seg_id=0)
    print(json.dumps(live, indent=2, ensure_ascii=False))
else:
    print("Live LLM call skipped. The deterministic validation exercises above are the default workshop path.")

## 10. Export an auditable record

The structured record can be saved as JSONL while preserving evidence, offsets, per-field status, model/provider metadata and review flags.

In [ ]:
from pathlib import Path

out_dir = Path("sh2026_outputs/annotations")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "journey_example.jsonl"

with out_path.open("w", encoding="utf-8") as fh:
    for row in result["journeys"]:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(out_path)

## 11. Take-away

The architecture is deliberately asymmetric:

**LLM proposes → software grounds and validates → human reviews where necessary**

not:

**LLM generates → database accepts**

The more representational freedom the model receives, the stronger the evidence and audit requirements must become.

**Next:** compare rules, contextual NLP, transformers and LLM outputs side by side, then measure disagreement and human correction burden.